## Invoice DocAI Project --- 05 Summary & Comparison

Final summary notebook: loads all prediction CSVs, computes metrics,
and produces comparison tables and visualisations for the project presentation.

## Plan
1. Setup & imports
2. Load all prediction CSVs (OCR / Donut x clean / corrupted)
3. Compute metrics per pipeline x condition
4. Main comparison table
5. Visualisation 1 --- Grouped bar chart (F1 by field)
6. Visualisation 2 --- Degradation chart (clean -> corrupted)
7. Visualisation 3 --- Exact match for date & total
8. Error analysis (OCR baseline)
9. Model comparison summary text
10. Save final summary CSV

In [ ]:
# =============================================================================
# Cell 1 --- Setup
# =============================================================================
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Mount Google Drive in Colab (if available)
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass


# --- Resolve PROJECT_ROOT (same pattern as other notebooks) -----------------
def is_project_root(p: Path) -> bool:
    return (
        (p / 'data' / 'sroie' / 'processed' / 'manifest_val.csv').exists()
        and (p / 'notebooks').exists()
        and (p / 'src').exists()
    )


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd / 'invoice_docai',
        Path('/content/invoice_docai'),
        Path('/content/drive/MyDrive/invoice_docai'),
        Path('/content/drive/MyDrive/ORC/invoice_docai'),
        Path('/content/drive/MyDrive/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/workspace/invoice_docai'),
        Path('/mnt/c/Yandex.Disk/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai')
            if Path('/mnt/c').exists() else None,
        Path(r'c:\\Yandex.Disk\\Yandex.Disk\\ML Neimark\\From OCR 2 Transformers\\invoice_docai'),
    ]
    for p in candidates:
        if p is None:
            continue
        try:
            p = p.resolve()
        except Exception:
            continue
        if is_project_root(p):
            return p
        child = p / 'invoice_docai'
        if child.is_dir() and is_project_root(child):
            return child

    # Fallback recursive search
    search_roots = [
        Path('/content/drive/MyDrive'),
        Path('/content/drive/.shortcut-targets-by-id'),
        Path('/content/drive'),
        Path('/content'),
        cwd,
    ]
    for base in search_roots:
        if not base.exists():
            continue
        try:
            for pattern in ['invoice_docai', 'invoice_docai*', '*invoice*doc*ai*']:
                for p in base.rglob(pattern):
                    if p.is_dir() and is_project_root(p):
                        return p.resolve()
        except Exception:
            pass
    raise FileNotFoundError(
        f'Cannot locate invoice_docai project root.  cwd={cwd}'
    )


PROJECT_ROOT = resolve_project_root()
OUTPUT_ROOT  = PROJECT_ROOT / 'v2' / 'outputs'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'sroie' / 'processed' / 'manifest_val.csv'

# Add src to path so we can import docai_utils
for src_dir in [PROJECT_ROOT / 'v2' / 'src', PROJECT_ROOT / 'src']:
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

from docai_utils import evaluate, normalize_vendor, normalize_date, normalize_total

# Matplotlib defaults
mpl.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'OUTPUT_ROOT  = {OUTPUT_ROOT}')
print(f'MANIFEST     = {MANIFEST_PATH}')

In [ ]:
# =============================================================================
# Cell 2 --- Load all prediction CSVs
# =============================================================================
PRED_FILES = {
    ('OCR',   'clean'):     'ocr_predictions_clean.csv',
    ('OCR',   'corrupted'): 'ocr_predictions_corrupted.csv',
    ('Donut', 'clean'):     'donut_predictions_clean.csv',
    ('Donut', 'corrupted'): 'donut_predictions_corrupted.csv',
}

predictions: dict = {}   # (pipeline, condition) -> DataFrame

for (pipeline, condition), fname in PRED_FILES.items():
    fpath = OUTPUT_ROOT / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        predictions[(pipeline, condition)] = df
        print(f'  Loaded {fname:40s}  ({len(df)} rows)')
    else:
        warnings.warn(f'  Missing: {fpath}')

# Load ground-truth manifest
manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest.dropna(subset=['image_path']).reset_index(drop=True)
print(f'\nGround-truth manifest: {len(manifest)} samples')
print(f'Available prediction sets: {list(predictions.keys())}')

In [ ]:
# =============================================================================
# Cell 3 --- Compute metrics for every available pipeline x condition
# =============================================================================
summary_rows = []

# Ground-truth columns needed by evaluate()
gt_cols = ['id', 'gt_vendor', 'gt_date', 'gt_total']

for (pipeline, condition), pred_df in predictions.items():
    metrics_df, errors_df = evaluate(manifest[gt_cols], pred_df)

    row = {'Pipeline': pipeline, 'Condition': condition}

    for _, m in metrics_df.iterrows():
        field = m['field']
        if field == 'micro':
            row['Micro_F1'] = round(m['f1'], 4)
        else:
            fname = field.capitalize()
            row[f'{fname}_P']     = round(m['precision'], 4)
            row[f'{fname}_R']     = round(m['recall'], 4)
            row[f'{fname}_F1']    = round(m['f1'], 4)
            row[f'{fname}_Exact'] = round(m['exact_match'], 4)

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

# Reorder columns for readability
col_order = [
    'Pipeline', 'Condition',
    'Vendor_P', 'Vendor_R', 'Vendor_F1', 'Vendor_Exact',
    'Date_P',   'Date_R',   'Date_F1',   'Date_Exact',
    'Total_P',  'Total_R',  'Total_F1',  'Total_Exact',
    'Micro_F1',
]
col_order = [c for c in col_order if c in summary.columns]
summary = summary[col_order]

print(f'Summary table: {summary.shape}')
summary

In [ ]:
# =============================================================================
# Cell 4 --- Display the main comparison table (styled)
# =============================================================================
def highlight_best(s):
    """Highlight the maximum value in each numeric column green."""
    if s.dtype in ('float64', 'float32'):
        is_best = s == s.max()
        return ['background-color: #d4edda' if v else '' for v in is_best]
    return ['' for _ in s]

try:
    styled = (
        summary.style
        .apply(highlight_best, axis=0)
        .format({
            c: '{:.2%}' for c in summary.columns
            if c not in ('Pipeline', 'Condition')
        })
        .set_caption('Pipeline Comparison: Precision / Recall / F1 / Exact Match')
    )
    display(styled)
except Exception:
    # Fallback for environments without rich display
    print(summary.to_string(index=False))

In [ ]:
# =============================================================================
# Cell 5 --- Visualisation 1: Grouped bar chart of F1 by field
# =============================================================================
FIELDS = ['Vendor', 'Date', 'Total']
f1_cols = [f'{f}_F1' for f in FIELDS]

# Build a label for each row: "Pipeline (condition)"
labels = summary.apply(lambda r: f"{r['Pipeline']} ({r['Condition']})", axis=1).tolist()

# Colours and hatching to distinguish clean vs corrupted
COLORS = {
    ('OCR',   'clean'):     '#4c78a8',
    ('OCR',   'corrupted'): '#9ecae1',
    ('Donut', 'clean'):     '#f58518',
    ('Donut', 'corrupted'): '#fdbe85',
}
HATCHES = {
    'clean':     '',
    'corrupted': '//',
}

x = np.arange(len(FIELDS))
n_bars = len(summary)
width = 0.8 / max(n_bars, 1)

fig, ax = plt.subplots(figsize=(10, 5))
for i, (idx, row) in enumerate(summary.iterrows()):
    key = (row['Pipeline'], row['Condition'])
    vals = [row.get(c, 0) for c in f1_cols]
    offset = (i - n_bars / 2 + 0.5) * width
    bars = ax.bar(
        x + offset, vals, width,
        label=labels[i],
        color=COLORS.get(key, f'C{i}'),
        hatch=HATCHES.get(row['Condition'], ''),
        edgecolor='black', linewidth=0.5,
    )
    # Annotate each bar with the value
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{v:.2f}',
            ha='center', va='bottom', fontsize=8,
        )

ax.set_xticks(x)
ax.set_xticklabels(FIELDS)
ax.set_ylabel('F1 Score')
ax.set_title('F1 by Field --- Pipeline x Condition')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'fig_f1_by_field.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_f1_by_field.png')

In [ ]:
# =============================================================================
# Cell 6 --- Visualisation 2: Degradation chart (clean -> corrupted)
# =============================================================================
pipelines_with_both = []
for pipeline in summary['Pipeline'].unique():
    conds = summary.loc[summary['Pipeline'] == pipeline, 'Condition'].tolist()
    if 'clean' in conds and 'corrupted' in conds:
        pipelines_with_both.append(pipeline)

if pipelines_with_both:
    degrad_rows = []
    for pipeline in pipelines_with_both:
        clean_row = summary[(summary['Pipeline'] == pipeline) & (summary['Condition'] == 'clean')].iloc[0]
        corr_row  = summary[(summary['Pipeline'] == pipeline) & (summary['Condition'] == 'corrupted')].iloc[0]
        for field in FIELDS:
            f1_clean = clean_row.get(f'{field}_F1', 0)
            f1_corr  = corr_row.get(f'{field}_F1', 0)
            degrad_rows.append({
                'Pipeline': pipeline,
                'Field': field,
                'F1_Clean': f1_clean,
                'F1_Corrupted': f1_corr,
                'Degradation': f1_clean - f1_corr,
            })
    degrad_df = pd.DataFrame(degrad_rows)

    x = np.arange(len(FIELDS))
    n_pipes = len(pipelines_with_both)
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    pipe_colors = {'OCR': '#4c78a8', 'Donut': '#f58518'}

    for i, pipeline in enumerate(pipelines_with_both):
        vals = degrad_df.loc[degrad_df['Pipeline'] == pipeline, 'Degradation'].values
        offset = (i - n_pipes / 2 + 0.5) * width
        bars = ax.bar(
            x + offset, vals, width,
            label=pipeline,
            color=pipe_colors.get(pipeline, f'C{i}'),
            edgecolor='black', linewidth=0.5,
        )
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{v:+.3f}',
                ha='center', va='bottom', fontsize=9,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(FIELDS)
    ax.set_ylabel('F1 Degradation (clean - corrupted)')
    ax.set_title('Robustness: F1 Drop under Messenger-Grade Corruption')
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / 'fig_degradation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fig_degradation.png')
else:
    print('Degradation chart skipped: need both clean and corrupted for at least one pipeline.')

In [ ]:
# =============================================================================
# Cell 7 --- Visualisation 3: Exact match for Date & Total
# =============================================================================
CRITICAL_FIELDS = ['Date', 'Total']
exact_cols = [f'{f}_Exact' for f in CRITICAL_FIELDS]

x = np.arange(len(CRITICAL_FIELDS))
n_bars = len(summary)
width = 0.8 / max(n_bars, 1)

fig, ax = plt.subplots(figsize=(8, 5))
for i, (idx, row) in enumerate(summary.iterrows()):
    key = (row['Pipeline'], row['Condition'])
    vals = [row.get(c, 0) for c in exact_cols]
    offset = (i - n_bars / 2 + 0.5) * width
    bars = ax.bar(
        x + offset, vals, width,
        label=labels[i],
        color=COLORS.get(key, f'C{i}'),
        hatch=HATCHES.get(row['Condition'], ''),
        edgecolor='black', linewidth=0.5,
    )
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{v:.1%}',
            ha='center', va='bottom', fontsize=9,
        )

ax.set_xticks(x)
ax.set_xticklabels(CRITICAL_FIELDS)
ax.set_ylabel('Exact Match Accuracy')
ax.set_title('Exact Match on Financially Critical Fields (Date & Total)')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'fig_exact_match.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_exact_match.png')

In [ ]:
# =============================================================================
# Cell 8 --- Error analysis (OCR baseline)
# =============================================================================
# Pick the first available OCR prediction set for error analysis
ocr_key = None
for k in [('OCR', 'clean'), ('OCR', 'corrupted')]:
    if k in predictions:
        ocr_key = k
        break

if ocr_key is not None:
    pred_df = predictions[ocr_key]
    _, errors_df = evaluate(manifest[['id', 'gt_vendor', 'gt_date', 'gt_total']], pred_df)

    # --- 8a: distribution of number of wrong fields per document ------------
    # Rebuild num_wrong_fields for ALL docs (not just error rows)
    merged_all = manifest[['id', 'gt_vendor', 'gt_date', 'gt_total']].merge(
        pred_df, on='id', how='inner',
    )
    for col in ['gt_vendor', 'pred_vendor']:
        merged_all[col] = merged_all[col].fillna('').map(normalize_vendor)
    for col in ['gt_date', 'pred_date']:
        merged_all[col] = merged_all[col].fillna('').map(normalize_date)
    for col in ['gt_total', 'pred_total']:
        merged_all[col] = merged_all[col].fillna('').map(normalize_total)

    merged_all['n_wrong'] = (
        (merged_all['gt_vendor'] != merged_all['pred_vendor']).astype(int)
        + (merged_all['gt_date'] != merged_all['pred_date']).astype(int)
        + (merged_all['gt_total'] != merged_all['pred_total']).astype(int)
    )

    wrong_dist = merged_all['n_wrong'].value_counts().sort_index()
    print(f'=== Error Analysis --- OCR baseline ({ocr_key[1]}) ===')
    print(f'\nDocuments with N wrong fields:')
    for n_wrong, count in wrong_dist.items():
        pct = count / len(merged_all) * 100
        print(f'  {n_wrong} wrong fields: {count:>4d} docs  ({pct:.1f}%)')

    # Bar chart of wrong-field distribution
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(wrong_dist.index.astype(str), wrong_dist.values, color='#e45756', edgecolor='black', linewidth=0.5)
    for xi, v in zip(wrong_dist.index, wrong_dist.values):
        ax.text(str(xi), v + 1, str(v), ha='center', va='bottom', fontsize=10)
    ax.set_xlabel('Number of Wrong Fields')
    ax.set_ylabel('Number of Documents')
    ax.set_title(f'OCR Baseline ({ocr_key[1]}): Wrong Fields per Document')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / 'fig_error_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    # --- 8b: most common failure mode per field ----------------------------
    print('\nMost common failure mode per field:')
    for field in ['vendor', 'date', 'total']:
        gt_col = f'gt_{field}'
        pred_col = f'pred_{field}'
        wrong_mask = merged_all[gt_col] != merged_all[pred_col]
        n_wrong_field = wrong_mask.sum()
        n_total = len(merged_all)
        pct = n_wrong_field / n_total * 100 if n_total else 0

        # Classify errors
        wrong_rows = merged_all[wrong_mask]
        n_empty_pred = (wrong_rows[pred_col] == '').sum()
        n_empty_gt   = (wrong_rows[gt_col] == '').sum()
        n_mismatch   = n_wrong_field - n_empty_pred - n_empty_gt

        print(f'\n  {field.upper()} --- {n_wrong_field}/{n_total} wrong ({pct:.1f}%)')
        if n_wrong_field > 0:
            print(f'    - Prediction empty (missed):     {n_empty_pred} ({n_empty_pred/n_wrong_field*100:.0f}%)')
            print(f'    - Ground truth empty (spurious):  {n_empty_gt} ({n_empty_gt/n_wrong_field*100:.0f}%)')
            print(f'    - Both non-empty but mismatched: {n_mismatch} ({n_mismatch/n_wrong_field*100:.0f}%)')
else:
    print('Error analysis skipped: no OCR predictions available.')

In [ ]:
# =============================================================================
# Cell 9 --- Model comparison summary text
# =============================================================================
print('=' * 70)
print('MODEL COMPARISON SUMMARY')
print('=' * 70)

# --- 9a: Which pipeline wins on each field (clean condition)? ---------------
clean_rows = summary[summary['Condition'] == 'clean']
if len(clean_rows) >= 2:
    print('\n--- Best pipeline per field (clean images) ---')
    for field in FIELDS:
        col = f'{field}_F1'
        if col in clean_rows.columns:
            best_idx = clean_rows[col].idxmax()
            best_pipe = clean_rows.loc[best_idx, 'Pipeline']
            best_val  = clean_rows.loc[best_idx, col]
            other_val = clean_rows.loc[clean_rows.index != best_idx, col].values
            other_str = ', '.join(f'{v:.4f}' for v in other_val)
            print(f'  {field:>8s}:  {best_pipe} wins  (F1={best_val:.4f}  vs  {other_str})')
elif len(clean_rows) == 1:
    pipe = clean_rows.iloc[0]['Pipeline']
    print(f'\n  Only one pipeline available on clean: {pipe}')
    for field in FIELDS:
        col = f'{field}_F1'
        if col in clean_rows.columns:
            print(f'    {field:>8s}  F1 = {clean_rows.iloc[0][col]:.4f}')
else:
    print('\n  No clean-condition results available.')

# --- 9b: Which pipeline is more robust? ------------------------------------
if pipelines_with_both:
    print('\n--- Robustness (average F1 degradation across fields) ---')
    for pipeline in pipelines_with_both:
        sub = degrad_df[degrad_df['Pipeline'] == pipeline]
        avg_deg = sub['Degradation'].mean()
        print(f'  {pipeline:>8s}:  avg degradation = {avg_deg:+.4f}')
    if len(pipelines_with_both) >= 2:
        avg_degs = {
            p: degrad_df[degrad_df['Pipeline'] == p]['Degradation'].mean()
            for p in pipelines_with_both
        }
        most_robust = min(avg_degs, key=avg_degs.get)
        print(f'  --> Most robust: {most_robust} (smallest average degradation)')
else:
    print('\n  Robustness comparison skipped (need clean + corrupted for a pipeline).')

# --- 9c: Latency comparison ------------------------------------------------
print('\n--- Latency comparison ---')
latency_found = False
for (pipeline, condition), pred_df in predictions.items():
    if 'latency' in pred_df.columns or 'latency_s' in pred_df.columns:
        lat_col = 'latency_s' if 'latency_s' in pred_df.columns else 'latency'
        avg_lat = pred_df[lat_col].mean()
        print(f'  {pipeline} ({condition}): {avg_lat:.1f} s/doc (from predictions CSV)')
        latency_found = True
if not latency_found:
    print('  No latency column found in prediction files.')
    print('  Typical values from project runs (CPU):')
    print('    OCR  (EasyOCR + heuristic extraction): ~36 s/doc')
    print('    Donut (end-to-end Transformer):        ~40 s/doc')

print('\n' + '=' * 70)

In [ ]:
# =============================================================================
# Cell 10 --- Save final summary CSV
# =============================================================================
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_ROOT / 'results_summary.csv'
summary.to_csv(out_path, index=False)
print(f'Saved summary table to: {out_path}')
print(f'Shape: {summary.shape}')
summary